#Unofficial [LeetArxiv](https://leetarxiv.substack.com/p/gumbel-sinkhorn-neural-sort) implementation of the paper *Learning Latent Permutations with Gumbel-Sinkhorn Networks*

The 2018 paper *Learning Latent Permutations with Gumbel-Sinkhorn Network* (Mena et al., 2018) introduces *Gumbel-Sinkhorn networks*: neural networks that learn to sort items by differentiation.


The complete writeup is available [here](https://leetarxiv.substack.com/p/gumbel-sinkhorn-neural-sort)

Gumbel-Sinkhorn networks build on the fact that permutations can be represented by matrices.
The authors are further inspired by:
1. The [reparametarization trick](https://leetarxiv.substack.com/p/stable-diffusion-from-scratch-1?utm_source=publication-search) from diffusion models.
2. The [1967 Sinkhorn matrix](https://leetarxiv.substack.com/p/sinkhorn-knopp-algorithm-24d?utm_source=publication-search) balancing algorithm.
3. [Belief propagation](https://leetarxiv.substack.com/p/sinkhorn-solves-sudoku?utm_source=publication-search) an obscure alternative to backprop that uses Sinkhorn balancing to solve Sudoku.

First, we’ll code two fundamental concepts introduced in (Mena et al., 2018):

1.   **Sinkhorn operator**: a differentiable approximation of a permutation.
2.   **Gumbel-Sinkhorn distribution**: an analog of the Gumbel Softmax distribution built on the [reparameterization trick](https://leetarxiv.substack.com/p/stable-diffusion-from-scratch-1?utm_source=publication-search) like in stable diffusion.








## Sinkhorn Operator
This is section 2.1 in the [LeetArxiv article](https://leetarxiv.substack.com/p/gumbel-sinkhorn-neural-sort).

The Sinkhorn operator is simply the [1967 Sinkhorn matrix](https://leetarxiv.substack.com/p/sinkhorn-knopp-algorithm-24d?utm_source=publication-search) balancing algorithm but divisions are replaced with logarithms and subtraction (for numerical stability).

We test that the row and cols sum to 1 as expected.

In [1]:
import torch
def SinkhornOperator(squareMatrix, sinkhornIterations):
    """
    Args:
      squareMatrix: a square matrix of shape [N, N], or a tensor of [batchSize, N, N]
      sinkhornIterations: number of sinkhorn iterations, usually ~20
    Returns:
      A 3D tensor of close-to-doubly-stochastic matrices (2D tensors are
        converted to 3D tensors with batchSizeequals to 1)
    """
    assert squareMatrix.shape[-1] == squareMatrix.shape[-2], \
        f"Last two dimensions must be equal, got {tuple(squareMatrix.shape)}"
    for iteration in range(sinkhornIterations):
        squareMatrix = squareMatrix - torch.logsumexp(squareMatrix, -1, keepdim=True)
        squareMatrix = squareMatrix - torch.logsumexp(squareMatrix, -2, keepdim=True)
    return squareMatrix.exp() #Remove logs

def TestSinkhornOperator():
  #Create a random square matrix.
  matrixDimension = 10
  sinkhornIterations = 12
  squareMatrix = torch.rand((matrixDimension, matrixDimension))
  #Apply the sinkhorn operator
  sinkhornResult = SinkhornOperator(torch.log(squareMatrix), sinkhornIterations)
  #Ensure all rows sum to 1.
  assert torch.allclose(sinkhornResult.sum(dim=0), torch.ones(sinkhornResult.shape[0]))
  #Ensure all columns sum to 1.
  assert torch.allclose(sinkhornResult.sum(dim=1), torch.ones(sinkhornResult.shape[1]))
  print(f"Original matrix :\n{squareMatrix}")
  print(f"Balanced matrix :\n{sinkhornResult}")

if __name__ == "__main__":
    TestSinkhornOperator()


Original matrix :
tensor([[0.3822, 0.1644, 0.5889, 0.9373, 0.8124, 0.0101, 0.4182, 0.9982, 0.1221,
         0.2263],
        [0.7682, 0.8496, 0.6960, 0.7168, 0.5759, 0.6849, 0.3907, 0.4597, 0.0725,
         0.4783],
        [0.3724, 0.0948, 0.6971, 0.3615, 0.1568, 0.9185, 0.5774, 0.4565, 0.8716,
         0.6975],
        [0.1321, 0.1834, 0.6492, 0.2009, 0.4487, 0.1179, 0.6300, 0.8116, 0.3008,
         0.5165],
        [0.8281, 0.3511, 0.6795, 0.6836, 0.2163, 0.0184, 0.4169, 0.6309, 0.1690,
         0.3114],
        [0.4823, 0.8928, 0.5040, 0.8472, 0.0277, 0.3790, 0.5029, 0.5120, 0.3651,
         0.6147],
        [0.4289, 0.3905, 0.6885, 0.7051, 0.6677, 0.9707, 0.8665, 0.7547, 0.5291,
         0.4564],
        [0.0043, 0.2808, 0.4461, 0.8507, 0.5136, 0.4977, 0.2199, 0.2802, 0.5415,
         0.3968],
        [0.9225, 0.4097, 0.6511, 0.1818, 0.1429, 0.3723, 0.8650, 0.7000, 0.4854,
         0.6652],
        [0.2690, 0.5030, 0.7652, 0.7054, 0.9410, 0.2217, 0.6350, 0.9035, 0.4302,
         0

## Gumbel-Sinkhorn Distribution
This is section 2.2 in the [LeetArxiv article](https://leetarxiv.substack.com/p/gumbel-sinkhorn-neural-sort).

The *Gumbel-Sinkhorn distribution* is the distribution obtained by applying the Sinkhorn-operator on an unnormalized assignment probability matrix to which we add Gumbel noise.

The concept's borrowed from Stable Diffusion's [reparameterization trick](https://leetarxiv.substack.com/p/stable-diffusion-from-scratch-1?utm_source=publication-search).


In [2]:
def GenerateGumbelNoise(tensorShape, device='cpu', eps=1e-20):
    """
    Args:
      tensorShape: list of tensor dimensions
      eps: small float to avoid division by zero
    """
    noise = torch.rand(tensorShape, device=device)
    return -torch.log(-torch.log(noise + eps) + eps)

def GumbelSinkhorn(sinkhornProbabilities, tau, sinkhornIterations):
    """Generate a Gumbel-Sinkhorn matrix at timestep tau.
    Args:
      sinkhornProbabilities: square matrix of log probabilities obtained from SinkhornOperator.
      tau: Temperature parameter. Think of it like the timestep parameter in diffusion forward process
           Small tau is little noise. Big tau is super noisy.
    """
    gumbelNoise = GenerateGumbelNoise(sinkhornProbabilities.shape, device=sinkhornProbabilities.device)
    #Apply the Sinkhorn operator
    noiseAtTimestepTau = SinkhornOperator((sinkhornProbabilities + gumbelNoise)/tau, sinkhornIterations)
    return noiseAtTimestepTau

def TestTau():
  #Create a random square matrix.
  matrixDimension = 4
  sinkhornIterations = 20
  squareMatrix = torch.rand((matrixDimension, matrixDimension), requires_grad=True)
  #Generate noisy versions
  smallTauResult = GumbelSinkhorn(squareMatrix, tau=0.01, sinkhornIterations=sinkhornIterations)
  bigTauResult   = GumbelSinkhorn(squareMatrix, tau=11,   sinkhornIterations=sinkhornIterations)

  #Test gradients
  X = torch.rand((matrixDimension, matrixDimension), requires_grad=True)
  P = GumbelSinkhorn(X*1000, tau=18, sinkhornIterations=2000)
  P.sum().backward()
  print(f"smallTauResult : Matrix entries are close to whole numbers and resemble a permutation matrix somewhat\n{smallTauResult}\n")
  print(f"bigTauResult   : Barely  resembles a permutation matrix\n{bigTauResult}\n")
  print(f"X Gradients    : It has gradients (lol you might see zeros bcoz the grads are super small)\n{X.grad}\n")

if __name__ == "__main__":
    TestTau()

smallTauResult : Matrix entries are close to whole numbers and resemble a permutation matrix somewhat
tensor([[1.0000e+00, 1.6093e-17, 4.4584e-26, 0.0000e+00],
        [0.0000e+00, 9.7502e-01, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 9.8999e-40, 0.0000e+00, 9.9992e-01],
        [7.6950e-12, 2.4975e-02, 1.0000e+00, 7.6426e-05]],
       grad_fn=<ExpBackward0>)

bigTauResult   : Barely  resembles a permutation matrix
tensor([[0.2552, 0.2649, 0.2338, 0.2461],
        [0.2486, 0.2679, 0.2530, 0.2305],
        [0.2336, 0.2379, 0.2805, 0.2480],
        [0.2625, 0.2293, 0.2327, 0.2754]], grad_fn=<ExpBackward0>)

X Gradients    : It has gradients (lol you might see zeros bcoz the grads are super small)
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])



## Gumbel-Sinkhorn Networks
This is section 2.3 in the [LeetArxiv article](https://leetarxiv.substack.com/p/gumbel-sinkhorn-neural-sort).

*Gumbel-Sinkhorn networks* build on the Sinkhorn-operator and Gumbel-Sinkhorn sampler.


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def matching(logAlpha):
  """Greedy assignment for evaluation"""
  probs = torch.exp(logAlpha)
  n = logAlpha.shape[0]
  assignment = torch.zeros_like(probs)
  for i in range(n):
    remainingCols = (assignment.sum(dim=0) == 0)
    if remainingCols.sum() == 0:
      break
    best_col = torch.argmax(probs[i] * remainingCols.float())
    assignment[i, best_col] = 1
  return assignment

class NumberGenerator(Dataset):
  def __init__(self, numberListLength, noOfLists, minValue=0, maxValue=1, seed=1):
    self.numberListLength = numberListLength
    torch.manual_seed(seed)
    self.X = torch.zeros(noOfLists, numberListLength).uniform_(minValue, maxValue)
    self.sortedX, self.permutation = self.X.sort(1)

  def __getitem__(self, idx):
    return self.X[idx], self.sortedX[idx], self.permutation[idx]

  def __len__(self):
    return len(self.X)

class NumberSortingNetwork(nn.Module):
  def __init__(self, numberListLength, tau=1.0, sinkhornIterations=20):
    super().__init__()
    self.numberListLength = numberListLength
    self.tau = tau
    self.sinkhornIterations = sinkhornIterations

    self.encoder = nn.Sequential(
      nn.Linear(numberListLength, numberListLength * 2),
      nn.ReLU(),
      nn.Linear(numberListLength * 2, numberListLength * 4),
      nn.ReLU(),
      nn.Linear(numberListLength * 4, numberListLength * numberListLength)
    )

  def forward(self, x):
    batchSize = x.shape[0]
    logitsFlattened = self.encoder(x)
    logits = logitsFlattened.view(batchSize, self.numberListLength, self.numberListLength)
    probs = torch.exp(logits - torch.logsumexp(logits, dim=-1, keepdim=True))

    if self.training:
      permutationMatrices = []
      for i in range(batchSize):
        P = GumbelSinkhorn(probs[i], tau=self.tau, sinkhornIterations=self.sinkhornIterations)
        permutationMatrices.append(P)
      P = torch.stack(permutationMatrices)
    else:
      permutationMatrices = []
      for i in range(batchSize):
        P = matching(logits[i])
        permutationMatrices.append(P)
      P = torch.stack(permutationMatrices)
    sortedX = torch.bmm(P, x.unsqueeze(-1)).squeeze(-1)
    return sortedX, P

# Training
numberListLength = 5
batchSize = 32
epochs = 50
learningRate = 1e-3
trainDataset = NumberGenerator(numberListLength, 500)
trainLoader = DataLoader(trainDataset, batch_size=batchSize, shuffle=True)
network = NumberSortingNetwork(numberListLength, tau=0.5, sinkhornIterations=20)
optimizer = torch.optim.Adam(network.parameters(), lr=learningRate)

network.train()
for epoch in range(epochs):
  totalLoss = 0
  for X, sortedX, _ in trainLoader:
    sortedPrediction, _ = network(X)
    loss = F.mse_loss(sortedPrediction, sortedX)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    totalLoss += loss.item()
  print(f"Epoch {epoch+1}/{epochs}, Loss: {totalLoss/len(trainLoader):.6f}")

# Test
print("\nTesting:")
network.eval()
testDataset = NumberGenerator(numberListLength, 5, seed=420)
with torch.no_grad():
  for i in range(5):
    X, sortedX, _ = testDataset[i]
    X = X.unsqueeze(0)
    sortedPrediction, P = network(X)

    print(f"\nOriginal: {X.squeeze().tolist()}")
    print(f"Predicted sorted: {sortedPrediction.squeeze().tolist()}")
    print(f"Actual sorted: {sortedX.tolist()}")
    print(f"Permutation matrix:\n{P.squeeze().numpy().round(2)}")

Epoch 1/50, Loss: 0.090421
Epoch 2/50, Loss: 0.095273
Epoch 3/50, Loss: 0.094168
Epoch 4/50, Loss: 0.093884
Epoch 5/50, Loss: 0.095240
Epoch 6/50, Loss: 0.092008
Epoch 7/50, Loss: 0.094545
Epoch 8/50, Loss: 0.090081
Epoch 9/50, Loss: 0.093812
Epoch 10/50, Loss: 0.089087
Epoch 11/50, Loss: 0.088694
Epoch 12/50, Loss: 0.093075
Epoch 13/50, Loss: 0.091368
Epoch 14/50, Loss: 0.088067
Epoch 15/50, Loss: 0.089241
Epoch 16/50, Loss: 0.090421
Epoch 17/50, Loss: 0.088694
Epoch 18/50, Loss: 0.087907
Epoch 19/50, Loss: 0.083260
Epoch 20/50, Loss: 0.087714
Epoch 21/50, Loss: 0.080586
Epoch 22/50, Loss: 0.082756
Epoch 23/50, Loss: 0.081492
Epoch 24/50, Loss: 0.078511
Epoch 25/50, Loss: 0.080917
Epoch 26/50, Loss: 0.080953
Epoch 27/50, Loss: 0.078570
Epoch 28/50, Loss: 0.076991
Epoch 29/50, Loss: 0.078676
Epoch 30/50, Loss: 0.077668
Epoch 31/50, Loss: 0.075959
Epoch 32/50, Loss: 0.073842
Epoch 33/50, Loss: 0.079902
Epoch 34/50, Loss: 0.080849
Epoch 35/50, Loss: 0.074145
Epoch 36/50, Loss: 0.076669
E